In [1]:
!pip install pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Define the input and output files
# --------------------------------------------------

sales_file = Path("Sales.csv")
output_file = Path("Daily_Sales.csv")

# --------------------------------------------------
# 2. Load the sales data
# --------------------------------------------------

if not sales_file.exists():
    raise FileNotFoundError(
        "Sales.csv was not found. Place this script "
        "in the same folder as Sales.csv."
    )

sales = pd.read_csv(sales_file)

print("Sales data loaded successfully!")
print("Original number of rows:", len(sales))
print("Available columns:", sales.columns.tolist())

# --------------------------------------------------
# 3. Check that the required columns exist
# --------------------------------------------------

required_columns = [
    "Invoice_Date",
    "Invoice_No",
    "Quantity",
    "Total_Sales",
    "Profit"
]

missing_columns = [
    column for column in required_columns
    if column not in sales.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        "Check the column names in Sales.csv."
    )

# --------------------------------------------------
# 4. Clean and prepare the data
# --------------------------------------------------

# Convert dates to datetime format
sales["Invoice_Date"] = pd.to_datetime(
    sales["Invoice_Date"],
    errors="coerce"
)

# Convert numeric columns to numbers
numeric_columns = ["Quantity", "Total_Sales", "Profit"]

for column in numeric_columns:
    sales[column] = pd.to_numeric(
        sales[column],
        errors="coerce"
    )

# Remove rows missing essential values
sales = sales.dropna(
    subset=[
        "Invoice_Date",
        "Invoice_No",
        "Quantity",
        "Total_Sales",
        "Profit"
    ]
)

# Keep the date only, without the time
sales["Invoice_Date"] = sales["Invoice_Date"].dt.date

# --------------------------------------------------
# 5. Aggregate sales by day
# --------------------------------------------------

daily_sales = (
    sales.groupby("Invoice_Date", as_index=False)
    .agg(
        Daily_Revenue=("Total_Sales", "sum"),
        Daily_Profit=("Profit", "sum"),
        Units_Sold=("Quantity", "sum"),
        Total_Orders=("Invoice_No", "nunique")
    )
)

# Calculate daily profit margin as a percentage
daily_sales["Profit_Margin_Percentage"] = (
    daily_sales["Daily_Profit"]
    .div(daily_sales["Daily_Revenue"].replace(0, pd.NA))
    .mul(100)
).round(2)

# Sort chronologically
daily_sales = daily_sales.sort_values("Invoice_Date")

# --------------------------------------------------
# 6. Export the result to CSV
# --------------------------------------------------

daily_sales.to_csv(output_file, index=False)

# --------------------------------------------------
# 7. Display a summary
# --------------------------------------------------

print("\nDaily sales file created successfully!")
print("Output file:", output_file.resolve())
print("Number of daily records:", len(daily_sales))

print("\nFirst five rows:")
print(daily_sales.head())

print("\nDaily sales summary:")
print(
    daily_sales[
        [
            "Daily_Revenue",
            "Daily_Profit",
            "Units_Sold",
            "Total_Orders"
        ]
    ].describe()
)



Sales data loaded successfully!
Original number of rows: 45324
Available columns: ['Sales_ID', 'Invoice_No', 'Invoice_Date', 'Store_ID', 'Product_ID', 'Customer_ID', 'Quantity', 'Unit_Price', 'Discount_Amount', 'Total_Sales', 'Cost_Per_Unit', 'Payment_Method', 'Line_Cost', 'Gross_Sales', 'Profit']

Daily sales file created successfully!
Output file: C:\Users\Admin\Desktop\FreshChoice_Analytics\Notebooks\Daily_Sales.csv
Number of daily records: 365

First five rows:
  Invoice_Date  Daily_Revenue  Daily_Profit  Units_Sold  Total_Orders  \
0   2025-01-01      163960.25      51481.10         222            64   
1   2025-01-02      154639.28      44140.44         214            62   
2   2025-01-03      178047.23      51218.36         231            68   
3   2025-01-04      179599.29      54542.05         239            67   
4   2025-01-05      188281.56      51975.07         297            89   

   Profit_Margin_Percentage  
0                     31.40  
1                     28.54  
2